16h. Ngưỡng hiệu chỉnh của (c) lgbm_neighbor_mean trên val, CPU (GĐ H, trước tag freeze-v1)

In [ ]:
import glob, json, os, resource, subprocess, sys, time
from contextlib import contextmanager
from pathlib import Path

ALLOW_GPU = False
WORK = "/kaggle/working"
REPO = "/tmp/gnn-iot-ids"
INFO = {"steps_s": {}, "device": "CPU", "gpu": None}
T_START = time.time()


def peak_ram_gb():
    kb = max(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss, resource.getrusage(resource.RUSAGE_CHILDREN).ru_maxrss)
    return round(kb / 2**20, 2)


def save_info():
    INFO["total_s"] = round(time.time() - T_START, 1)
    INFO["peak_ram_gb"] = peak_ram_gb()
    INFO["gpu_hours"] = round(INFO["total_s"] / 3600, 3) if INFO["device"] == "GPU" else 0
    with open(f"{WORK}/run_info.json", "w", encoding="utf-8") as f:
        json.dump(INFO, f, ensure_ascii=False, indent=1)


@contextmanager
def step(name):
    t0 = time.time()
    try:
        yield
    finally:
        INFO["steps_s"][name] = round(time.time() - t0, 1)
        save_info()
        print(f"--- {name}: {INFO['steps_s'][name]} s, RAM đỉnh tới nay {INFO['peak_ram_gb']} GB")


def run(cmd, ok=(0,), **kw):
    r = subprocess.run(cmd, capture_output=True, text=True, **kw)
    print(r.stdout[-9000:], end="")
    print(r.stderr[-2500:], end="", file=sys.stderr)
    print(f"[mã thoát {r.returncode}]")
    if r.returncode not in ok:
        raise RuntimeError(f"Lệnh lỗi: {' '.join(cmd)}")
    return r


def find(pattern):
    """Đường dẫn đầu tiên khớp glob đệ quy dưới /kaggle/input (báo lỗi rõ nếu không có)."""
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    assert hits, f"Không thấy {pattern} trong /kaggle/input (đã gắn đủ nguồn dữ liệu chưa?)"
    return hits[0]


with step("clone_va_cai_goi"):
    run(["rm", "-rf", REPO])
    run(["git", "clone", "--depth", "1", "https://github.com/minucedev/gnn-iot-ids.git", REPO])
    INFO["commit"] = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
    with open(f"{REPO}/requirements.txt") as f:
        reqs = [l.strip() for l in f if l.strip() and not l.startswith("#")]
    keep = [r for r in reqs if r.split("==")[0].lower() not in {"torch", "jupyter", "ipykernel"}]
    run([sys.executable, "-m", "pip", "install", "-q", *keep])

import torch
if torch.cuda.is_available():
    INFO["device"], INFO["gpu"] = "GPU", torch.cuda.get_device_name(0)
INFO["cpu_count"] = os.cpu_count()
save_info()
print("Commit:", INFO["commit"], "| máy:", INFO["device"], INFO["gpu"])
assert ALLOW_GPU or INFO["device"] == "CPU", "Chưa được phép dùng GPU: tắt GPU trong cài đặt notebook"

## pytest toàn bộ (dừng nếu lỗi)

In [ ]:
with step("pytest"):
    # test tái lập hai lần chạy độc lập không ổn định trên GPU (scatter không tất định); nó đã pass trên CPU ở notebook 16t và 16a
    r = run([sys.executable, "-m", "pytest", "-q", "--deselect", "tests/test_hybrid.py::test_run_hybrid_end_to_end_both_modes_val_only_reproducible"] if INFO["device"] == "GPU" else [sys.executable, "-m", "pytest", "-q"], cwd=REPO)
    INFO["pytest"] = r.stdout.strip().splitlines()[-1]

## Đường dẫn đầu vào

In [ ]:
OLD = f"{REPO}/reports/kaggle"          # kết quả ablation GĐ G đã kéo về repo (cấu hình LightGBM của (c) và F1 val để đối chiếu)
SRC = {"nf_ton_iot_v3": ("gnn-iot-ids-16d-ablation-ton"), "nf_bot_iot_v3": ("gnn-iot-ids-16a-ablation-bot")}
INFO["datasets"] = {}
for DS in SRC:
    INFO["datasets"][DS] = os.path.dirname(find(f"{DS}/split.json"))    # output của notebook 04 (ToN) và 11 (BoT)
save_info()
print(json.dumps(INFO["datasets"], indent=1))

## Ngưỡng của (c) `lgbm_neighbor_mean` trên val, cả 2 bộ, 3 seed
Huấn luyện lại (c) đúng như bản ablation GĐ G, đối chiếu F1 val với `ablation_cd_<bộ>.csv` (dừng nếu lệch > 5e-4), chọn ngưỡng theo quy tắc
của `src/threshold.py` (nhỏ nhất sao cho FPR val ≤ 4,08%), lưu mô hình. Không đọc tập test.

In [ ]:
for DS, abl in SRC.items():
    with step(f"threshold_c_{DS}"):
        run([sys.executable, "-I", "scripts/run_threshold_c.py", INFO["datasets"][DS], DS, WORK,
             f"search_json={OLD}/{abl}/ablation_search_{DS}.json", f"ablation_csv={OLD}/{abl}/ablation_cd_{DS}.csv"], cwd=REPO)

In [ ]:
thr = {}
for DS in SRC:
    thr.update(json.load(open(f"{WORK}/thresholds_c_{DS}.json", encoding="utf-8")))
INFO["repro_f1_diff_max_abs"] = {ds: max(abs(c["repro_f1_diff"]) for c in v["lgbm_neighbor_mean"].values()) for ds, v in thr.items()}
INFO["thresholds"] = {ds: {s: c["threshold"] for s, c in v["lgbm_neighbor_mean"].items()} for ds, v in thr.items()}
save_info()
print(json.dumps({"repro_f1_diff_max_abs": INFO["repro_f1_diff_max_abs"], "thresholds": INFO["thresholds"]}, indent=1))
print(open(f"{WORK}/run_info.json", encoding="utf-8").read())